In [7]:
%pip install -q faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 11.2 MB/s eta 0:00:0000:0100:01


In [1]:
import os

os.environ["HF_TOKEN"] = input()

In [2]:
from datasets import load_dataset

dataset = load_dataset("pameydorke/arcanum-cross-platform-queries-synthetic")

dataset["train"]

README.md:   0%|          | 0.00/585 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.93MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  743kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4612 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1153 [00:00<?, ? examples/s]

Dataset({
    features: ['query', 'document_title', 'document_text', 'document_link', 'query_type', 'difficulty'],
    num_rows: 4612
})

In [3]:
from sentence_transformers import SentenceTransformer
from datasets import Dataset

retriever = SentenceTransformer("pameydorke/arcanum-cross-platform-retriever")

train_split = dataset["train"]

corpus_texts = list(dict.fromkeys(train_split["document_text"]))
corpus_dataset = Dataset.from_dict({"text": corpus_texts})

corpus_embeddings = retriever.encode(
    corpus_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True
)

modules.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/284 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/62.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Batches:   0%|          | 0/37 [00:00<?, ?it/s]

In [8]:
from sentence_transformers.util import mine_hard_negatives
from datasets import Dataset

train_dataset = dataset["train"].select_columns(["query", "document_text"])
train_dataset = train_dataset.rename_columns({
    "query": "anchor",
    "document_text": "positive"
})

hard_train_dataset = mine_hard_negatives(
    dataset=train_dataset,
    model=retriever,
    corpus=corpus_texts,
    num_negatives=4,
    margin=0,
    range_min=0,
    range_max=100,
    sampling_strategy="top",
    batch_size=128,
    output_format="labeled-pair",
    use_faiss=True,
)

print(f"Original dataset size: {len(train_dataset)}")
print(f"Enriched dataset size: {len(hard_train_dataset)}")
print(hard_train_dataset[0])

Found 4603 unique queries out of 4612 total queries.
Found an average of 1.002 positives per query.


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Querying FAISS index: 100%|██████████| 1/1 [00:00<00:00,  5.25it/s]


Negative candidates mined, preparing dataset...
Metric       Positive       Negative     Difference
Count           4,612         18,271               
Mean           0.5250         0.3928         0.1360
Median         0.5395         0.3846         0.1190
Std            0.1380         0.0984         0.1063
Min           -0.0667         0.1023         0.0000
25%            0.4351         0.3195         0.0426
50%            0.5395         0.3846         0.1190
75%            0.6261         0.4587         0.2091
Max            0.8639         0.7730         0.5469
Skipped 18,856 potential negatives (3.98%) due to the absolute_margin of 0.
Could not find enough negatives for 177 samples (0.77%). Consider adjusting the range_max and absolute_margin parameters if you'd like to find more valid negatives.
Original dataset size: 4612
Enriched dataset size: 22847
{'anchor': 'compound bow explosives requirement', 'positive': 'Pyrotechnic Bow. Pyrotechnic Bow. The Pyrotechnic Bow Schematic can be 

In [41]:
unique_queries = list(set(hard_train_dataset["anchor"]))
print(f"Total unique queries: {len(unique_queries)}")

Total unique queries: 4603


In [42]:
from sklearn.model_selection import train_test_split

train_queries, val_queries = train_test_split(
    unique_queries, test_size=0.1, random_state=42
)

In [45]:
train_set, val_set = set(train_queries), set(val_queries)
train_dataset = hard_train_dataset.filter(lambda x: x["anchor"] in train_set)
val_dataset   = hard_train_dataset.filter(lambda x: x["anchor"] in val_set)

print(f"Train rows: {len(train_dataset)}, queries: {len(train_set)}")
print(f"Val rows:   {len(val_dataset)},   queries: {len(val_set)}")

Filter:   0%|          | 0/22847 [00:00<?, ? examples/s]

Filter:   0%|          | 0/22847 [00:00<?, ? examples/s]

Train rows: 20569, queries: 4142
Val rows:   2278,   queries: 461


In [47]:
train_dataset[0], train_dataset[161]

({'anchor': 'main choices arcana',
  'positive': 'Original Post: What aspects of Arcanum do you wish other games would adopt?. For me, I really liked the way you could just bash doors down. It always bugged me in other games, where you\'d have a giant hammer and godlike strength, or a rocket launcher, yet perfectly simple doors would deter you. \n\n[Case in point](http://www.google.com/imgres?newwindow=1&amp;sa=X&amp;espvd=210&amp;es_sm=93&amp;tbm=isch&amp;tbnid=9ZxAC40ZRiR_qM%3A&amp;imgrefurl=http%3A%2F%2Fjoyreactor.com%2Fpost%2F618863&amp;docid=5we7LcmNonM21M&amp;imgurl=http%3A%2F%2Fimg1.joyreactor.com%2Fpics%2Fpost%2Ffunny-pictures-fallout-auto-530501.jpeg&amp;w=780&amp;h=624&amp;ei=o_viUqmuN4jwoASF4YK4DA&amp;zoom=1&amp;ved=0CGAQhBwwBA&amp;iact=rc&amp;dur=186&amp;page=1&amp;start=0&amp;ndsp=39&amp;biw=1680&amp;bih=959)\n\nNot Arcanum! The only fault I can find with the system is Virgil\'s overenthusiastic response to the idea of breaking down doors. \n\nWhat aspects of Arcanum would

In [49]:
from collections import defaultdict

grouped = defaultdict(lambda: {"positive": [], "negative": []})

for anchor, document, label in zip(
    val_dataset["anchor"], val_dataset["positive"], val_dataset["label"]
):
    key = "positive" if int(label) == 1 else "negative"
    grouped[anchor][key].append(document)

# Only keep queries that have at least one positive AND one negative
eval_samples = [
    {"query": q, **docs}
    for q, docs in grouped.items()
    if docs["positive"] and docs["negative"]
]

print(f"Evaluation queries: {len(eval_samples)}")

Evaluation queries: 454


In [50]:
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

evaluator = CrossEncoderRerankingEvaluator(
    samples=eval_samples,
    at_k=10,
    name="val-hard-negatives",
    batch_size=64,
    show_progress_bar=True,
    always_rerank_positives=True,
)

In [ ]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("BAAI/bge-reranker-v2-m3", num_labels=1)

In [51]:
results = evaluator(model)
print(results)

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

{'val-hard-negatives_map': 0.8845814977973568, 'val-hard-negatives_mrr@10': 0.8845814977973568, 'val-hard-negatives_ndcg@10': 0.9141341657696732}


In [ ]:
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from datasets import Dataset

train_dataset_formatted = Dataset.from_dict({
    "query": [p["anchor"] for p in train_dataset],
    "document": [p["positive"] for p in train_dataset],
    "label": [p["label"] for p in train_dataset]
})

val_dataset_formatted = Dataset.from_dict({
    "query": [p["anchor"] for p in val_dataset],
    "document": [p["positive"] for p in val_dataset],
    "label": [p["label"] for p in val_dataset]
})

train_loss = BinaryCrossEntropyLoss(model)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [53]:
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments
from transformers import EarlyStoppingCallback

args = CrossEncoderTrainingArguments(
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    logging_steps=50,

    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_val-hard-negatives_ndcg@10",
    greater_is_better=True,

    use_cpu=False,
    bf16=True,
    report_to="none",
)

trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset_formatted,
    eval_dataset=val_dataset_formatted,
    loss=train_loss,
        callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,          # stop after 3 evals w/o improvement
            early_stopping_threshold=0.001,     # minimum improvement to count
        )
    ],
)

In [ ]:
trainer.train()